# Copyright

<PRE>
Jelen iPython notebook a Budapesti Műszaki és Gazdaságtudományi Egyetemen tartott
"MI ágensek és multiágensrendszerek fejlesztése" tantárgy segédanyagaként készült.

A notebook bármely részének újra felhasználása, publikálása csak a szerzők írásos beleegyezése esetén megegengedett.

2026 (c) Potyók Csaba (potyok kukac mit pont bme pont hu)
</PRE>

# MCP eszközök fejlesztése

A gyakorlat betekintést nyújt abba, hogyan lehet MCP szervereket készíteni, amelyeket az LLM-alapú ágensek fel tudnak használni a probléma megoldásuk során. A gyakorlat során megnézzük, hogy lehet MCP szerverekhez eszközöket létrehozni.

# Szükséges könyvtárak

A gyakorlat során a [Pydantic AI](https://ai.pydantic.dev/) könyvtárat használjuk, mivel ennek segítségével típusbiztosan és egyszerűen tudunk eszközöket definiálni.

In [1]:
%pip install pydantic-ai sqlmodel "pydantic-ai-slim[fastmcp]" fastmcp logfire ipython sqlalchemy pywin32 --quiet

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1
[notice] To update, run: python.exe -m pip install --upgrade pip


# Környezeti változók beállítása

Az API kulcs biztonságos kezelése érdekében a Google Colab `Titkos kulcsok` funkcióját használjuk, ahol biztonságosan hozzáférhetünk a notebook futtatása során a használt API kulcsainkhoz.

In [2]:
import os
from getpass import getpass

mistral_api_key = os.getenv('MISTRAL_API_KEY') or os.getenv('Mistral_API_KEY')
if not mistral_api_key:
    mistral_api_key = getpass('Enter your Mistral API key: ')

os.environ['MISTRAL_API_KEY'] = mistral_api_key
os.environ['Mistral_API_KEY'] = mistral_api_key

# 1. Naptárbejegyzéseket kezelő MCP szerver

Ebben a szakaszban készítünk egy egyszerű MCP szervert, amely képes kiolvasni bejegyzéseket a felhasználó naptárából. A naptár elkészítéséhez a `sqlmodel` modult fogjuk használni annak érdekében, hogy egy valódi adatbázist tudjuk Colab környezetben szimulálni.

A naptárbejegyzéseket tartalmazó adatbázis előkészítése.

In [3]:
from sqlmodel import SQLModel, Field, create_engine, Session, select, JSON, Column, Relationship, or_, and_
from sqlalchemy.orm import selectinload
from typing import List, Optional, Literal, Union
from datetime import datetime, timedelta

Definiáljuk a `CalendarEntry` adatmodellt, amely tartalmazza a következő mezőket: `title`, `description`, `start_time` és `end_time`. Adjunk meg minden mezőhöz egy rövid leírást. Az időpontok kezeléséhez használjuk a `datetime` modult.

In [4]:
class CalendarEntry(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str = Field(description="Event title")
    description: str = Field(description="Event description")
    start_time: datetime = Field(description="Event start time")
    end_time: datetime = Field(description="Event end time")

Inicializáljuk az adatbázist!

In [5]:
calendar_sqlite_url = "sqlite:///calendar.db"
calendar_engine = create_engine(calendar_sqlite_url)

In [6]:
SQLModel.metadata.create_all(calendar_engine)

In [7]:
# Ha összeakadnának a tábla verziók, ezzel lehet törölni a sémákat
# SQLModel.metadata.clear()

## Példa adatok

Adjunk meg néhány példa bejegyzést, amelyet a naptárunk fog tartalmazni!

In [8]:
with Session(calendar_engine) as session:
    example_events = [
        CalendarEntry(
            title="Project Kickoff",
            description="Initial meeting for the new MCP tutorial series.",
            start_time=datetime(2026, 1, 20, 9, 0),
            end_time=datetime(2026, 1, 20, 11, 0)
        ),
        CalendarEntry(
            title="Quick Sync",
            description="Daily stand-up with the development team.",
            start_time=datetime(2026, 1, 21, 10, 0),
            end_time=datetime(2026, 1, 21, 10, 15)
        ),
        CalendarEntry(
            title="Deep Work: Coding",
            description="Implementing the Pydantic AI agent logic.",
            start_time=datetime(2026, 1, 23, 13, 0),
            end_time=datetime(2026, 1, 23, 17, 0)
        ),
        CalendarEntry(
            title="Client Feedback Session",
            description="Reviewing the first draft of the database schema.",
            start_time=datetime(2026, 1, 26, 15, 0),
            end_time=datetime(2026, 1, 26, 16, 30)
        ),
        CalendarEntry(
            title="Gym Workout",
            description="Morning strength training session.",
            start_time=datetime(2026, 1, 29, 8, 0),
            end_time=datetime(2026, 1, 29, 9, 15)
        ),
        CalendarEntry(
            title="Strategy Planning",
            description="Quarterly goals and roadmap definition.",
            start_time=datetime(2026, 2, 2, 10, 0),
            end_time=datetime(2026, 2, 2, 14, 0)
        ),
        CalendarEntry(
            title="Technical Interview",
            description="Hiring interview for the Junior Dev position.",
            start_time=datetime(2026, 2, 5, 11, 0),
            end_time=datetime(2026, 2, 5, 12, 0)
        ),
        CalendarEntry(
            title="Lunch with Mentor",
            description="Discussing career growth and MCP integration.",
            start_time=datetime(2026, 2, 9, 12, 30),
            end_time=datetime(2026, 2, 9, 14, 0)
        ),
        CalendarEntry(
            title="Bug Scrub",
            description="Reviewing open issues in the GitHub repository.",
            start_time=datetime(2026, 2, 12, 16, 0),
            end_time=datetime(2026, 2, 12, 16, 45)
        ),
        CalendarEntry(
            title="Valentine's Dinner",
            description="Private dinner reservation at the Italian bistro.",
            start_time=datetime(2026, 2, 14, 19, 0),
            end_time=datetime(2026, 2, 14, 21, 30)
        ),
        CalendarEntry(
            title="Architecture Review",
            description="Evaluating the system design with senior leads.",
            start_time=datetime(2026, 2, 17, 14, 0),
            end_time=datetime(2026, 2, 17, 16, 0)
        ),
        CalendarEntry(
            title="Coffee Break Sync",
            description="Casual chat with the design team.",
            start_time=datetime(2026, 2, 20, 10, 30),
            end_time=datetime(2026, 2, 20, 10, 45)
        ),
        CalendarEntry(
            title="Workshop: SQLModel",
            description="Internal training on using SQLModel for AI apps.",
            start_time=datetime(2026, 2, 24, 9, 0),
            end_time=datetime(2026, 2, 24, 12, 0)
        ),
        CalendarEntry(
            title="Market Research",
            description="Analyzing competitors in the AI agent space.",
            start_time=datetime(2026, 2, 27, 15, 0),
            end_time=datetime(2026, 2, 27, 17, 45)
        ),
        CalendarEntry(
            title="Code Review",
            description="Peer review of the latest pull requests.",
            start_time=datetime(2026, 3, 2, 11, 0),
            end_time=datetime(2026, 3, 2, 11, 45)
        ),
        CalendarEntry(
            title="Product Demo",
            description="Showcasing the MCP server to stakeholders.",
            start_time=datetime(2026, 3, 5, 14, 0),
            end_time=datetime(2026, 3, 5, 15, 0)
        ),
        CalendarEntry(
            title="Language Lesson",
            description="Weekly advanced English conversation class.",
            start_time=datetime(2026, 3, 9, 18, 0),
            end_time=datetime(2026, 3, 9, 19, 0)
        ),
        CalendarEntry(
            title="Sprint Planning",
            description="Task allocation for the upcoming two weeks.",
            start_time=datetime(2026, 3, 11, 9, 0),
            end_time=datetime(2026, 3, 11, 12, 0)
        ),
        CalendarEntry(
            title="Status Update",
            description="Quick call to update management on progress.",
            start_time=datetime(2026, 3, 13, 16, 0),
            end_time=datetime(2026, 3, 13, 16, 15)
        ),
        CalendarEntry(
            title="Final Testing",
            description="End-to-end testing of the Colab environment.",
            start_time=datetime(2026, 3, 15, 10, 0),
            end_time=datetime(2026, 3, 15, 14, 0)
        )
    ]
    session.add_all(example_events)
    session.commit()

## MCP szerver elkészítése

Importáljuk a `pydantic_ai` modult, amelynek segítségével hozzuk létre az MCP szerverünket, majd az ágensünket!

In [9]:
from fastmcp import FastMCP

from pydantic_ai.toolsets.fastmcp import FastMCPToolset

Definiáljuk a `Calendar` nevű MCP szervert, továbbá adjuk hozzá a következő eszközöket, amelyeket az ágensünket fog majd használni!

In [10]:
calendar_mcp_server = FastMCP('Calendar')

Definiáljuk a `get_today` eszközt, amelynek a segítségével egy beégett dátumot kapunk vissza, amely a mai nap dátumaként fog szolgálni!

In [11]:
@calendar_mcp_server.tool()
def get_today():
  return datetime.now().date().isoformat()

Definiáljuk a `get_events_for_week` eszközt, amely képes egy adott `target_date` hetére eső összes naplóbejegyzést kilistázni.

In [12]:
@calendar_mcp_server.tool()
def get_events_for_week(target_date: datetime):
    start_of_week = (target_date - timedelta(days=target_date.weekday())).replace(hour=0, minute=0, second=0, microsecond=0)
    end_of_week = start_of_week + timedelta(days=7)

    with Session(calendar_engine) as session:
        statement = select(CalendarEntry).where(
            and_(
                CalendarEntry.start_time >= start_of_week,
                CalendarEntry.start_time < end_of_week,
            )
        ).order_by(CalendarEntry.start_time)
        events = session.exec(statement).all()

    return [
        {
            "id": event.id,
            "title": event.title,
            "description": event.description,
            "start_time": event.start_time.isoformat(),
            "end_time": event.end_time.isoformat(),
        }
        for event in events
    ]

Definiáljuk a `get_events_after_date` eszközt, amely egy adott `target_date` dátumhoz képest `days_offset` napra lévő eseményeket tudja kilistázni.

In [13]:
@calendar_mcp_server.tool()
def get_events_after_date(target_date: datetime, days_offset: int = 0):
  target_day = (target_date + timedelta(days=days_offset)).replace(hour=0, minute=0, second=0, microsecond=0)
  next_day = target_day + timedelta(days=1)

  with Session(calendar_engine) as session:
      statement = select(CalendarEntry).where(
          and_(
              CalendarEntry.start_time >= target_day,
              CalendarEntry.start_time < next_day,
          )
      ).order_by(CalendarEntry.start_time)
      events = session.exec(statement).all()

  return [
      {
          "id": event.id,
          "title": event.title,
          "description": event.description,
          "start_time": event.start_time.isoformat(),
          "end_time": event.end_time.isoformat(),
      }
      for event in events
  ]

Definiáljuk a `insert_event` eszközt, amivel az ágens fel tud venni egy új naptárbejegyzést!

In [14]:
@calendar_mcp_server.tool()
def insert_event(title: str, description: str, start_time: datetime, end_time: datetime):
  with Session(calendar_engine) as session:
      event = CalendarEntry(
          title=title,
          description=description,
          start_time=start_time,
          end_time=end_time,
      )
      session.add(event)
      session.commit()
      session.refresh(event)

  return {
      "id": event.id,
      "title": event.title,
      "description": event.description,
      "start_time": event.start_time.isoformat(),
      "end_time": event.end_time.isoformat(),
  }

In [15]:
toolset = FastMCPToolset(calendar_mcp_server)

## Ágens elkészítése

Importáljuk a szükséges modulokat, hogy létrehozzuk a naptár kezelésére szolgáló ágenst!

In [16]:
from pydantic import BaseModel
from typing import List, Dict

import logfire

from pydantic_ai import (
    Agent,
    RunContext,
    UsageLimits,
    ModelMessage,
    RunUsage
)

In [17]:
logfire.configure(send_to_logfire='if-token-present')
logfire.instrument_pydantic_ai()

In [18]:
import asyncio
from pydantic_ai.exceptions import ModelHTTPError


async def run_with_rate_limit_retry(agent, prompt, attempts: int = 3, base_delay: float = 2.0):
    last_error = None
    for attempt in range(attempts):
        try:
            return await agent.run(prompt)
        except ModelHTTPError as error:
            last_error = error
            if getattr(error, 'status_code', None) != 429 or attempt == attempts - 1:
                raise
            await asyncio.sleep(base_delay * (2 ** attempt))
    raise last_error

A modell kiválasztásánál fontos figyelembe venni, hogy a modell támogassa az eszközhasználatot (tool-use).

In [19]:
model_name = 'mistral:mistral-small-latest'

Definiáljuk a `calendar_agent` ágenst!

In [20]:
calendar_agent = Agent(
    model_name,
    system_prompt=(
        "You are a calendar management agent whose job is to answer questions asked by users based on calendar entries. ",
        "Check the calendar for today's date."
    ),
    toolsets=[toolset]
)

Próbáljuk ki az elkészített ágensünket, tegyünk fel neki kérdéseket!

In [21]:
result = await run_with_rate_limit_retry(calendar_agent, "What's my schedule for tomorrow?")

20:11:23.189 agent run
20:11:23.224   chat mistral-small-latest
20:11:23.274     chat_completion_v1_chat_completions_post
20:11:24.214   running tool: get_events_after_date
20:11:24.216     tools/call get_events_after_date
20:11:24.219   tools/call get_events_after_date
20:11:24.257   chat mistral-small-latest
20:11:24.282     chat_completion_v1_chat_completions_post
20:11:24.717   running tool: get_today
20:11:24.718     tools/call get_today
20:11:24.721   tools/call get_today
20:11:24.729   chat mistral-small-latest
20:11:24.754     chat_completion_v1_chat_completions_post
20:11:25.341   running tool: get_events_after_date
20:11:25.342     tools/call get_events_after_date
20:11:25.344   tools/call get_events_after_date
20:11:25.355   chat mistral-small-latest
20:11:25.380     chat_completion_v1_chat_completions_post


Amennyiben szeretnénk olvashatóbbá tenni az ágens válaszát, úgy használjuk az alábbi kódrészletet!

In [22]:
from IPython.display import display, Markdown

In [23]:
display(Markdown(result.output))

It looks like you don't have any events scheduled for tomorrow, **May 3, 2026**. Would you like to add something to your schedule?

Kérjük meg az ágenst, hogy vegyen fel nekünk egy új naptárbejegyzést, majd utána ellenőirzzük a művelet sikerességét!

In [24]:
result = await run_with_rate_limit_retry(calendar_agent, "Please schedule a one-hour yoga class for me tonight at 8 p.m.!")

20:11:26.087 agent run
20:11:26.106   chat mistral-small-latest
20:11:26.147     chat_completion_v1_chat_completions_post
20:11:26.889   running tool: insert_event
20:11:26.889     tools/call insert_event
20:11:26.891   tools/call insert_event
20:11:26.962   chat mistral-small-latest
20:11:26.987     chat_completion_v1_chat_completions_post


In [25]:
display(Markdown(result.output))

Your one-hour yoga class has been successfully scheduled for tonight at 8 p.m. Enjoy your session! 🧘‍♀️

In [26]:
result = await run_with_rate_limit_retry(calendar_agent, "What calendar entries do I have for today?")

20:11:27.909 agent run
20:11:27.926   chat mistral-small-latest
20:11:27.954     chat_completion_v1_chat_completions_post
20:11:28.267   running tool: get_today
20:11:28.268     tools/call get_today
20:11:28.270   tools/call get_today
20:11:28.280   chat mistral-small-latest
20:11:28.305     chat_completion_v1_chat_completions_post
20:11:28.837   running tool: get_events_after_date
20:11:28.839     tools/call get_events_after_date
20:11:28.841   tools/call get_events_after_date
20:11:28.857   chat mistral-small-latest
20:11:28.887     chat_completion_v1_chat_completions_post


In [27]:
display(Markdown(result.output))

You have no calendar entries for today, May 2nd, 2026. Would you like to add an event?

# 2. Mozi figyelő ágens

Ebben a szakaszban egy olyan MCP szervert készítünk, amely mozifilmeket és azok vetítéseit kezeli. Az ágens ezt a szervert használva tud lekérdezni információkat filmekről, valamint megtudni, hogy egyes filmeket mikor vetítenek a mozikban.

In [28]:
from enum import Enum

class MovieGenre(str, Enum):
  thriller = 'thriller'
  action = 'action'
  adventure = 'adventure'
  drama = 'drama'
  sci_fi = 'sci-fi'
  romantic = 'romantic'
  comedy = 'comedy'

Definiáljuk a `Movie` és a `MovieScreening` adatmodelleket! A `Movie` egy mozifilmet reprezentál, amelynek a következő mezői vannak: `title`, `short_description`, `genre` és `length`. A `MovieScreening` a vetítésekhez tartozó információkat írja le, ehhez `movie_id` és `start_time` mezőket definiál.

In [29]:
class Movie(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    title: str
    short_description: str
    genre: List[str] = Field(sa_column=Column(JSON))
    length: int
    screenings: List["MovieScreening"] = Relationship(back_populates="movie")


class MovieScreening(SQLModel, table=True):
    id: Optional[int] = Field(default=None, primary_key=True)
    movie_id: Optional[int] = Field(default=None, foreign_key="movie.id")
    start_time: datetime
    movie: Optional[Movie] = Relationship(back_populates="screenings")

Hozzunk létre olyan adatmodelleket, amelyek általunk megszabadott módon írják le az egyes filmekhez tartozó információkat! A `MovieInfo` modell csak `id` és `title` mezőket tartalmazza, míg a `DetailedMovieInfo` modell tartalmazza továbbá a `genre`, `length` és `screenings` attributúmokat is.

In [30]:
class MovieInfo(BaseModel):
    id: int
    title: str


class DetailedMovieInfo(MovieInfo):
    short_description: str
    genre: List[str]
    length: int
    screenings: List[datetime]

Definiáljuk az adatbázist ahol tárolni fogjuk ezeket az információkat!

In [31]:
movie_sqlite_url = "sqlite:///movie.db"
movie_engine = create_engine(movie_sqlite_url)

In [32]:
SQLModel.metadata.create_all(movie_engine)

## Példa adatok

Adjunk meg néhány példa filmet és vetítési alkalmat, amelyet az adatbázisunk fog tartalmazni!

In [33]:
with Session(movie_engine) as session:
    example_movies = [
        Movie(
            title="Interstellar",
            short_description="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.",
            genre=["sci-fi", "adventure"],
            length=169,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 20, 9, 0)),  # Ütközik: Project Kickoff
                MovieScreening(start_time=datetime(2026, 1, 22, 18, 30)),
                MovieScreening(start_time=datetime(2026, 1, 25, 20, 0)),
                MovieScreening(start_time=datetime(2026, 1, 29, 8, 0)),   # Ütközik: Gym Workout
                MovieScreening(start_time=datetime(2026, 2, 2, 10, 0)),   # Ütközik: Strategy Planning
                MovieScreening(start_time=datetime(2026, 2, 10, 19, 0)),
                MovieScreening(start_time=datetime(2026, 2, 15, 14, 0)),
                MovieScreening(start_time=datetime(2026, 3, 1, 16, 0))
            ]
        ),
        Movie(
            title="The Dark Knight",
            short_description="When the menace known as the Joker wreaks havoc and chaos on the people of Gotham.",
            genre=["action", "thriller"],
            length=152,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 21, 10, 0)),  # Ütközik: Quick Sync
                MovieScreening(start_time=datetime(2026, 1, 24, 21, 0)),
                MovieScreening(start_time=datetime(2026, 1, 28, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 5, 11, 0)),   # Ütközik: Technical Interview
                MovieScreening(start_time=datetime(2026, 2, 12, 16, 0)),  # Ütközik: Bug Scrub
                MovieScreening(start_time=datetime(2026, 2, 18, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 5, 14, 0)),    # Ütközik: Product Demo
                MovieScreening(start_time=datetime(2026, 3, 12, 20, 0))
            ]
        ),
        Movie(
            title="Superbad",
            short_description="Two co-dependent high school seniors are forced to deal with separation anxiety.",
            genre=["comedy"],
            length=113,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 23, 14, 0)),  # Átfedés: Deep Work
                MovieScreening(start_time=datetime(2026, 1, 27, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 1, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 9, 12, 30)),  # Ütközik: Lunch with Mentor
                MovieScreening(start_time=datetime(2026, 2, 14, 20, 0)),  # Átfedés: Valentine's Dinner
                MovieScreening(start_time=datetime(2026, 2, 22, 17, 0)),
                MovieScreening(start_time=datetime(2026, 3, 3, 21, 0))
            ]
        ),
        Movie(
            title="Inception",
            short_description="A thief who steals corporate secrets through the use of dream-sharing technology.",
            genre=["sci-fi", "action"],
            length=148,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 26, 15, 0)),  # Ütközik: Client Feedback
                MovieScreening(start_time=datetime(2026, 1, 31, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 6, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 17, 14, 0)),  # Ütközik: Architecture Review
                MovieScreening(start_time=datetime(2026, 2, 24, 9, 0)),   # Ütközik: Workshop SQLModel
                MovieScreening(start_time=datetime(2026, 3, 6, 22, 0)),
                MovieScreening(start_time=datetime(2026, 3, 11, 10, 0))   # Átfedés: Sprint Planning
            ]
        ),
        Movie(
            title="Parasite",
            short_description="Greed and class discrimination threaten the newly formed symbiotic relationship.",
            genre=["thriller", "drama"],
            length=132,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 20, 19, 0)),
                MovieScreening(start_time=datetime(2026, 1, 24, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 3, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 11, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 20, 10, 30)), # Ütközik: Coffee Break Sync
                MovieScreening(start_time=datetime(2026, 3, 2, 11, 0)),   # Ütközik: Code Review
                MovieScreening(start_time=datetime(2026, 3, 10, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 15, 11, 0))   # Átfedés: Final Testing
            ]
        ),
        Movie(
            title="La La Land",
            short_description="While navigating their careers in Los Angeles, a pianist and an actress fall in love.",
            genre=["romantic", "drama"],
            length=128,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 22, 20, 0)),
                MovieScreening(start_time=datetime(2026, 1, 28, 19, 0)),
                MovieScreening(start_time=datetime(2026, 2, 7, 21, 0)),
                MovieScreening(start_time=datetime(2026, 2, 14, 18, 0)),  # Átfedés: Valentine's Dinner
                MovieScreening(start_time=datetime(2026, 2, 21, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 28, 20, 0)),
                MovieScreening(start_time=datetime(2026, 3, 9, 18, 0)),   # Ütközik: Language Lesson
                MovieScreening(start_time=datetime(2026, 3, 14, 16, 0))
            ]
        ),
        Movie(
            title="The Grand Budapest Hotel",
            short_description="A writer encounters the owner of a high-class hotel, who tells him of his early years.",
            genre=["comedy", "adventure"],
            length=99,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 21, 14, 0)),
                MovieScreening(start_time=datetime(2026, 1, 25, 11, 0)),
                MovieScreening(start_time=datetime(2026, 1, 30, 17, 0)),
                MovieScreening(start_time=datetime(2026, 2, 8, 14, 0)),
                MovieScreening(start_time=datetime(2026, 2, 15, 10, 0)),
                MovieScreening(start_time=datetime(2026, 2, 23, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 4, 18, 0)),
                MovieScreening(start_time=datetime(2026, 3, 13, 16, 0))   # Ütközik: Status Update
            ]
        ),
        Movie(
            title="Mad Max: Fury Road",
            short_description="In a post-apocalyptic wasteland, a woman rebels against a tyrannical ruler.",
            genre=["action", "adventure"],
            length=120,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 23, 10, 0)),
                MovieScreening(start_time=datetime(2026, 1, 29, 21, 0)),
                MovieScreening(start_time=datetime(2026, 2, 4, 17, 0)),
                MovieScreening(start_time=datetime(2026, 2, 12, 19, 0)),
                MovieScreening(start_time=datetime(2026, 2, 19, 22, 0)),
                MovieScreening(start_time=datetime(2026, 2, 27, 15, 0)),  # Ütközik: Market Research
                MovieScreening(start_time=datetime(2026, 3, 8, 14, 0)),
                MovieScreening(start_time=datetime(2026, 3, 14, 21, 0))
            ]
        ),
        Movie(
            title="Dune",
            short_description="A noble family becomes embroiled in a war for control over the galaxy's most valuable asset.",
            genre=["sci-fi", "adventure"],
            length=155,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 24, 10, 0)),
                MovieScreening(start_time=datetime(2026, 1, 31, 14, 0)),
                MovieScreening(start_time=datetime(2026, 2, 10, 15, 0)),
                MovieScreening(start_time=datetime(2026, 2, 18, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 25, 17, 0)),
                MovieScreening(start_time=datetime(2026, 3, 7, 21, 0)),
                MovieScreening(start_time=datetime(2026, 3, 15, 9, 0))    # Átfedés: Final Testing
            ]
        ),
        Movie(
            title="Shutter Island",
            short_description="A U.S. Marshal investigates the disappearance of a murderer who escaped from a hospital.",
            genre=["thriller", "drama"],
            length=138,
            screenings=[
                MovieScreening(start_time=datetime(2026, 1, 20, 22, 0)),
                MovieScreening(start_time=datetime(2026, 1, 27, 20, 0)),
                MovieScreening(start_time=datetime(2026, 2, 2, 18, 0)),
                MovieScreening(start_time=datetime(2026, 2, 11, 21, 0)),
                MovieScreening(start_time=datetime(2026, 2, 21, 20, 0)),
                MovieScreening(start_time=datetime(2026, 3, 1, 14, 0)),
                MovieScreening(start_time=datetime(2026, 3, 8, 19, 0)),
                MovieScreening(start_time=datetime(2026, 3, 12, 17, 0))
            ]
        )
    ]
    session.add_all(example_movies)
    session.commit()

## MCP szerver elkészítése

Definiáljuk a `Movie` nevű MCP szerverünket, illetve készítsük el hozzá a szükséges eszközöket!

In [34]:
movie_mcp_server = FastMCP('Movie')

Definiáljuk a `get_title_of_movies` eszközt, amely kilistázza az adatbázisban tárolt összes film címét!

In [35]:
@movie_mcp_server.tool()
def get_title_of_movies():
  with Session(movie_engine) as session:
      movies = session.exec(select(Movie).order_by(Movie.title)).all()
  return [movie.title for movie in movies]

Definiáljuk a `get_screenings_for_movie` eszközt, amely egy konkrét film címéhez kikeresi az egyes vetítési alkalmait!

In [36]:
@movie_mcp_server.tool()
def get_screenings_for_movie(movie_title: str):
  with Session(movie_engine) as session:
      movie = session.exec(select(Movie).options(selectinload(Movie.screenings)).where(Movie.title == movie_title)).first()

  if movie is None:
      return {"error": f"Movie not found: {movie_title}"}

  return {
      "movie_id": movie.id,
      "title": movie.title,
      "screenings": [screening.start_time.isoformat() for screening in movie.screenings],
  }

Definiáljuk a `get_movie_info` eszközt, amely egy konkrét id-val rendelkező filmhez lekérdezi az összes információt!

In [37]:
@movie_mcp_server.tool()
def get_movie_info(movie_id: str) -> DetailedMovieInfo:
    with Session(movie_engine) as session:
        movie = session.exec(select(Movie).options(selectinload(Movie.screenings)).where(Movie.id == int(movie_id))).first()

    if movie is None:
        raise ValueError(f"Movie not found: {movie_id}")

    return DetailedMovieInfo(
        id=movie.id,
        title=movie.title,
        short_description=movie.short_description,
        genre=movie.genre,
        length=movie.length,
        screenings=[screening.start_time for screening in movie.screenings],
    )

Definiáljunk egy `search_movies_by_filter` eszközt, amely összetett lekérdezéseket tesz lehetővé! A `genres` paraméterében opcionálisan megadható egy műfaj lista (`or` művelettel összefűzve a feltételben), illetve a `target_date` paraméterében opcionálisan megadható dátum. A `response_format` lehetővé teszi, hogy az ágens válasza a kapott válasz részletezettségét.

In [38]:
@movie_mcp_server.tool()
def search_movies_by_filter(
    genres: Optional[List[Literal['thriller', 'action', 'adventure', 'drama', 'sci-fi', 'romantic', 'comedy']]] = None,
    target_date: Optional[datetime] = None,
    response_format: Literal['concise', 'detailed'] = "detailed"
    ) -> Union[List[MovieInfo], List[DetailedMovieInfo]]:
  with Session(movie_engine) as session:
    movies = session.exec(select(Movie).options(selectinload(Movie.screenings))).all()

  if target_date:
    target_day = target_date.date()
    movies = [
        movie
        for movie in movies
        if any(screening.start_time.date() == target_day for screening in movie.screenings)
    ]

  if genres:
    movies = [movie for movie in movies if any(genre in movie.genre for genre in genres)]

  if response_format == "concise":
    return [MovieInfo(title=movie.title, id=movie.id) for movie in movies]

  return [DetailedMovieInfo(
      title=movie.title,
      id=movie.id,
      short_description=movie.short_description,
      genre=movie.genre,
      length=movie.length,
      screenings=[screening.start_time for screening in movie.screenings]
  ) for movie in movies]

Próbáljuk ki az elkészített `search_movies_by_filter` működését!

In [39]:
search_movies_by_filter(target_date=datetime(2026, 1, 20))

[DetailedMovieInfo(id=1, title='Interstellar', short_description="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.", genre=['sci-fi', 'adventure'], length=169, screenings=[datetime.datetime(2026, 1, 20, 9, 0), datetime.datetime(2026, 1, 22, 18, 30), datetime.datetime(2026, 1, 25, 20, 0), datetime.datetime(2026, 1, 29, 8, 0), datetime.datetime(2026, 2, 2, 10, 0), datetime.datetime(2026, 2, 10, 19, 0), datetime.datetime(2026, 2, 15, 14, 0), datetime.datetime(2026, 3, 1, 16, 0)]),
 DetailedMovieInfo(id=5, title='Parasite', short_description='Greed and class discrimination threaten the newly formed symbiotic relationship.', genre=['thriller', 'drama'], length=132, screenings=[datetime.datetime(2026, 1, 20, 19, 0), datetime.datetime(2026, 1, 24, 15, 0), datetime.datetime(2026, 2, 3, 20, 0), datetime.datetime(2026, 2, 11, 18, 0), datetime.datetime(2026, 2, 20, 10, 30), datetime.datetime(2026, 3, 2, 11, 0), datetime.datetime(2026, 3, 10, 19, 

In [40]:
get_screenings_for_movie("Interstellar")

{'movie_id': 1,
 'title': 'Interstellar',
 'screenings': ['2026-01-20T09:00:00',
  '2026-01-22T18:30:00',
  '2026-01-25T20:00:00',
  '2026-01-29T08:00:00',
  '2026-02-02T10:00:00',
  '2026-02-10T19:00:00',
  '2026-02-15T14:00:00',
  '2026-03-01T16:00:00']}

In [41]:
get_movie_info("1")

DetailedMovieInfo(id=1, title='Interstellar', short_description="A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.", genre=['sci-fi', 'adventure'], length=169, screenings=[datetime.datetime(2026, 1, 20, 9, 0), datetime.datetime(2026, 1, 22, 18, 30), datetime.datetime(2026, 1, 25, 20, 0), datetime.datetime(2026, 1, 29, 8, 0), datetime.datetime(2026, 2, 2, 10, 0), datetime.datetime(2026, 2, 10, 19, 0), datetime.datetime(2026, 2, 15, 14, 0), datetime.datetime(2026, 3, 1, 16, 0)])

## Ágens elkészítése

Készítsük el a mozi kereső ágensünket!

In [42]:
toolset = FastMCPToolset(movie_mcp_server)

In [43]:
movie_agent = Agent(
    model_name,
    system_prompt=(
        "You are a movie search agent who can provide information about movies that meet the user's request. ",
    ),
    toolsets=[toolset]
)

## Ágens tesztelése

A következő szakaszban készítsünk pár példa hívást, amivel megvizsgálhatjuk mennyire jól működik együtt az ágens az általunk készített MCP szerverrel.

### 1. eset: "What are the names of the movies?"

In [44]:
result = await run_with_rate_limit_retry(movie_agent, "What are the names of the movies?")

20:11:31.748 agent run
20:11:31.764   chat mistral-small-latest
20:11:31.791     chat_completion_v1_chat_completions_post
20:11:32.412   running tool: get_title_of_movies
20:11:32.412     tools/call get_title_of_movies
20:11:32.414   tools/call get_title_of_movies
20:11:32.430   chat mistral-small-latest
20:11:32.456     chat_completion_v1_chat_completions_post


In [45]:
display(Markdown(result.output))

Here are some movies you might be interested in:

1. **Dune**
2. **Inception**
3. **Interstellar**
4. **La La Land**
5. **Mad Max: Fury Road**
6. **Parasite**
7. **Shutter Island**
8. **Superbad**
9. **The Dark Knight**
10. **The Grand Budapest Hotel**

In [46]:
get_title_of_movies()

['Dune',
 'Dune',
 'Dune',
 'Dune',
 'Inception',
 'Inception',
 'Inception',
 'Inception',
 'Interstellar',
 'Interstellar',
 'Interstellar',
 'Interstellar',
 'La La Land',
 'La La Land',
 'La La Land',
 'La La Land',
 'Mad Max: Fury Road',
 'Mad Max: Fury Road',
 'Mad Max: Fury Road',
 'Mad Max: Fury Road',
 'Parasite',
 'Parasite',
 'Parasite',
 'Parasite',
 'Shutter Island',
 'Shutter Island',
 'Shutter Island',
 'Shutter Island',
 'Superbad',
 'Superbad',
 'Superbad',
 'Superbad',
 'The Dark Knight',
 'The Dark Knight',
 'The Dark Knight',
 'The Dark Knight',
 'The Grand Budapest Hotel',
 'The Grand Budapest Hotel',
 'The Grand Budapest Hotel',
 'The Grand Budapest Hotel']

### 2. eset: "What adventure movies are showing at the moment?"

In [47]:
result = await run_with_rate_limit_retry(movie_agent, "What adventure movies are showing at the moment?")

20:11:33.459 agent run
20:11:33.475   chat mistral-small-latest
20:11:33.516     chat_completion_v1_chat_completions_post
20:11:33.966   running tool: search_movies_by_filter
20:11:33.967     tools/call search_movies_by_filter
20:11:33.969   tools/call search_movies_by_filter
20:11:34.053   chat mistral-small-latest
20:11:34.078     chat_completion_v1_chat_completions_post


In [48]:
display(Markdown(result.output))

Here are the adventure movies showing at the moment, along with their screening times:

---

### **1. Interstellar**
**Description:** A team of explorers travel through a wormhole in space in an attempt to ensure humanity's survival.
**Length:** 169 minutes
**Genres:** Sci-fi, Adventure
**Screenings:**
- **2026-01-20** 09:00
- **2026-01-22** 18:30
- **2026-01-25** 20:00
- **2026-01-29** 08:00
- **2026-02-02** 10:00
- **2026-02-10** 19:00
- **2026-02-15** 14:00
- **2026-03-01** 16:00

---

### **2. The Grand Budapest Hotel**
**Description:** A writer encounters the owner of a high-class hotel, who tells him of his early years.
**Length:** 99 minutes
**Genres:** Comedy, Adventure
**Screenings:**
- **2026-01-21** 14:00
- **2026-01-25** 11:00
- **2026-01-30** 17:00
- **2026-02-08** 14:00
- **2026-02-15** 10:00
- **2026-02-23** 19:00
- **2026-03-04** 18:00
- **2026-03-13** 16:00

---
### **3. Mad Max: Fury Road**
**Description:** In a post-apocalyptic wasteland, a woman rebels against a tyrannical ruler.
**Length:** 120 minutes
**Genres:** Action, Adventure
**Screenings:**
- **2026-01-23** 10:00
- **2026-01-29** 21:00
- **2026-02-04** 17:00
- **2026-02-12** 19:00
- **2026-02-19** 22:00
- **2026-02-27** 15:00
- **2026-03-08** 14:00
- **2026-03-14** 21:00

---
### **4. Dune**
**Description:** A noble family becomes embroiled in a war for control over the galaxy's most valuable asset.
**Length:** 155 minutes
**Genres:** Sci-fi, Adventure
**Screenings:**
- **2026-01-24** 10:00
- **2026-01-31** 14:00
- **2026-02-10** 15:00
- **2026-02-18** 20:00
- **2026-02-25** 17:00
- **2026-03-07** 21:00
- **2026-03-15** 09:00

### 3. eset: "Recommend some movies for me for the afternoon of March 7, 2026."

In [49]:
result = await run_with_rate_limit_retry(movie_agent, "Recommend some movies for me for the afternoon of March 7, 2026.")

20:11:38.354 agent run
20:11:38.376   chat mistral-small-latest
20:11:38.433     chat_completion_v1_chat_completions_post
20:11:39.070   running tool: search_movies_by_filter
20:11:39.071     tools/call search_movies_by_filter
20:11:39.074   tools/call search_movies_by_filter
20:11:39.109   chat mistral-small-latest
20:11:39.132     chat_completion_v1_chat_completions_post
20:11:39.697   running tool: get_screenings_for_movie
20:11:39.698     tools/call get_screenings_for_movie
20:11:39.700   tools/call get_screenings_for_movie
20:11:39.719   chat mistral-small-latest
20:11:39.746     chat_completion_v1_chat_completions_post


In [50]:
display(Markdown(result.output))

Here’s a movie recommendation for your afternoon of **March 7, 2026**:

### **Movie: *Dune***
- **Genre:** Sci-Fi, Adventure
- **Duration:** 155 minutes (2 hours 35 minutes)
- **Screening Time on March 7, 2026:** **9:00 PM** (Evening screening)
- **Description:** A noble family becomes embroiled in a war for control over the galaxy's most valuable asset.

Since the afternoon screenings are not available for this movie on March 7, 2026, you might want to consider:
1. **Checking for other movies** with afternoon screenings on that date.
2. **Attending the evening screening** of *Dune* if you're flexible with timing.

Would you like me to search for other movies with afternoon screenings on March 7, 2026? Let me know! 😊

In [51]:
search_movies_by_filter( target_date=datetime(2026, 3, 7))

[DetailedMovieInfo(id=9, title='Dune', short_description="A noble family becomes embroiled in a war for control over the galaxy's most valuable asset.", genre=['sci-fi', 'adventure'], length=155, screenings=[datetime.datetime(2026, 1, 24, 10, 0), datetime.datetime(2026, 1, 31, 14, 0), datetime.datetime(2026, 2, 10, 15, 0), datetime.datetime(2026, 2, 18, 20, 0), datetime.datetime(2026, 2, 25, 17, 0), datetime.datetime(2026, 3, 7, 21, 0), datetime.datetime(2026, 3, 15, 9, 0)]),
 DetailedMovieInfo(id=19, title='Dune', short_description="A noble family becomes embroiled in a war for control over the galaxy's most valuable asset.", genre=['sci-fi', 'adventure'], length=155, screenings=[datetime.datetime(2026, 1, 24, 10, 0), datetime.datetime(2026, 1, 31, 14, 0), datetime.datetime(2026, 2, 10, 15, 0), datetime.datetime(2026, 2, 18, 20, 0), datetime.datetime(2026, 2, 25, 17, 0), datetime.datetime(2026, 3, 7, 21, 0), datetime.datetime(2026, 3, 15, 9, 0)]),
 DetailedMovieInfo(id=29, title='Dune

## Saját próbálkozások

Írjatok még néhány kérést az ágens számára (esetleg további eszközöket definiálva) és vizsgáljátok meg miképp viselkedik az ágens!

In [52]:
result = await run_with_rate_limit_retry(movie_agent, "List the available movies and suggest one adventure movie for this week.")
display(Markdown(result.output))

20:11:41.233 agent run
20:11:41.247   chat mistral-small-latest
20:11:41.275     chat_completion_v1_chat_completions_post
20:11:41.788   running tool: get_title_of_movies
20:11:41.790     tools/call get_title_of_movies
20:11:41.790   running tool: search_movies_by_filter
20:11:41.791     tools/call search_movies_by_filter
20:11:41.795   tools/call get_title_of_movies
20:11:41.796   tools/call search_movies_by_filter
20:11:41.843   chat mistral-small-latest
20:11:41.873     chat_completion_v1_chat_completions_post
20:11:42.558   running tool: search_movies_by_filter
20:11:42.560     tools/call search_movies_by_filter
20:11:42.561   tools/call search_movies_by_filter
20:11:42.597   chat mistral-small-latest
20:11:42.622     chat_completion_v1_chat_completions_post


Here is the list of available movies:

- **Dune**
- **Inception**
- **Interstellar**
- **La La Land**
- **Mad Max: Fury Road**
- **Parasite**
- **Shutter Island**
- **Superbad**
- **The Dark Knight**
- **The Grand Budapest Hotel**

Since you requested a suggestion for an **adventure movie** for this week, I recommend:

### **Mad Max: Fury Road**
**Genre:** Adventure, Action
**Why?** It's a high-energy, visually stunning film with a gripping post-apocalyptic storyline. Perfect for an exciting movie night!

Would you like more details about this movie or its showtimes? 😊

# 3. Szorgalmi: MCP szerverek kombinálása

A feladat, hogy a gyakorlat során elkészített 'calendar_agent' és 'movie_agent' segítségével készítsünk egy olyan megoldást, ahol képesek vagyunk ezeket az ágenseket kooperatívan használni arra, hogy a kiválasztott filmet végül a naptárunkba is be tudjuk ütemezni. Azaz egy olyan asszisztenst kell készíteni, amely képes adott kritériumok alapján filmet ajánlani, majd azt a naptárba is felvenni, mint tevékenység a megfelelő paraméterekkel.

Az értékeléshez töltse fel egyetlen ZIP-állományként a kiegészített Jupyter Notebookot, illetve egy rövid PDF-beszámolót, amely bemutatja a kód működését, illetve a megvalósítás főbb lépéseit.

In [ ]:
import json
from typing import Any, Dict, Optional

def _extract_json_object(text: str) -> str:
    """JSON objektum kinyerése szövegből."""
    start_index = text.find("{")
    end_index = text.rfind("}")
    if start_index == -1 or end_index == -1 or end_index <= start_index:
        raise ValueError("The movie agent did not return a JSON object.")
    return text[start_index : end_index + 1]


async def recommend_and_schedule_movie(
    preferences: str,
    preferred_date: Optional[datetime] = None,
    calendar_event_title: str = "Movie night",
) -> Dict[str, Any]:
    movie_prompt = (
        "Use the movie tools to recommend one movie that best matches the user's request. "
        "Return ONLY valid JSON with the keys movie_title, screening_start, screening_end, and reason. "
        "Use ISO 8601 strings for the datetimes. "
        f"User request: {preferences}."
    )
    if preferred_date is not None:
        movie_prompt += f" Preferred date: {preferred_date.date().isoformat()}."

    movie_result = await run_with_rate_limit_retry(movie_agent, movie_prompt)
    recommendation = json.loads(_extract_json_object(movie_result.output))

    calendar_prompt = (
        f"Schedule a calendar event with title '{calendar_event_title}: {recommendation['movie_title']}', "
        f"description '{recommendation['reason']}', "
        f"start_time '{recommendation['screening_start']}', "
        f"end_time '{recommendation['screening_end']}'. "
        "If the time slot is busy, do not insert the event and explain the conflict briefly."
    )
    calendar_result = await run_with_rate_limit_retry(calendar_agent, calendar_prompt)

    return {
        "movie_recommendation": recommendation,
        "calendar_result": calendar_result.output,
    }

### Teszt 1: Sci-fi film ajánlása február 15-én

In [59]:
result = await recommend_and_schedule_movie(
    preferences="Recommend a sci-fi movie for me that's not too long.",
    preferred_date=datetime(2026, 2, 15),
    calendar_event_title="Movie night"
)

print("🎬 FILM AJÁNLÁS:")
print(json.dumps(result["movie_recommendation"], indent=2))
print("\n📅 NAPTÁR ÜTEMEZÉS EREDMÉNYE:")
print(result["calendar_result"])

20:41:16.430 agent run
20:41:16.447   chat mistral-small-latest
20:41:16.483     chat_completion_v1_chat_completions_post
20:41:17.190 agent run
20:41:17.202   chat mistral-small-latest
20:41:17.232     chat_completion_v1_chat_completions_post
20:41:18.071   running tool: insert_event
20:41:18.072     tools/call insert_event
20:41:18.075   tools/call insert_event
20:41:18.129   chat mistral-small-latest
20:41:18.152     chat_completion_v1_chat_completions_post
🎬 FILM AJÁNLÁS:
{
  "movie_title": "Interstellar",
  "screening_start": "2026-02-15T14:00:00",
  "screening_end": "2026-02-15T16:49:00",
  "reason": "It is a sci-fi movie, but it is quite long (169 min). No other options are available for the preferred date."
}

📅 NAPTÁR ÜTEMEZÉS EREDMÉNYE:
The event **"Movie night: Interstellar"** has been successfully scheduled for **February 15, 2026**, from **2:00 PM to 4:49 PM**.

Would you like to add any reminders or additional details to this event? 😊


### Teszt 2: Adventure film ajánlása

In [60]:
result = await recommend_and_schedule_movie(
    preferences="I want to watch an adventure movie this week that I'm free for.",
    preferred_date=datetime(2026, 1, 25),
    calendar_event_title="Adventure Cinema Night"
)

print("🎬 FILM AJÁNLÁS:")
print(json.dumps(result["movie_recommendation"], indent=2))
print("\n📅 NAPTÁR ÜTEMEZÉS EREDMÉNYE:")
print(result["calendar_result"])

20:41:27.105 agent run
20:41:27.123   chat mistral-small-latest
20:41:27.155     chat_completion_v1_chat_completions_post
20:41:27.834 agent run
20:41:27.844   chat mistral-small-latest
20:41:27.873     chat_completion_v1_chat_completions_post
20:41:28.739   running tool: insert_event
20:41:28.742     tools/call insert_event
20:41:28.745   tools/call insert_event
20:41:28.802   chat mistral-small-latest
20:41:28.829     chat_completion_v1_chat_completions_post
🎬 FILM AJÁNLÁS:
{
  "movie_title": "Interstellar",
  "screening_start": "2026-01-25T20:00:00",
  "screening_end": "2026-01-25T22:49:00",
  "reason": "It is an adventure movie with a screening available on your preferred date, **2026-01-25**."
}

📅 NAPTÁR ÜTEMEZÉS EREDMÉNYE:
Your event **"Adventure Cinema Night: Interstellar"** has been successfully scheduled!

- **Date:** January 25, 2026
- **Time:** 8:00 PM - 10:49 PM
- **Description:** It is an adventure movie with a screening available on your preferred date, **2026-01-25**.

